In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: "%.4f" % x)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

with open("../../data/customer_churn/meta.json") as f:
    meta = json.load(f)

df = pd.read_csv("../../data/customer_churn/churn_cleaned.csv")
print("Data loaded:", df.shape)
print("Churn rate :", round(df["Churn_Binary"].mean()*100, 2), "%")

Data loaded: (7043, 22)
Churn rate : 26.54 %


In [2]:
print("ENCODING CATEGORICAL FEATURES")
print()

binary_map = {"Yes": 1, "No": 0, "Female": 1, "Male": 0}
binary_cols = ["gender", "Partner", "Dependents", "PhoneService",
               "PaperlessBilling", "Churn"]
for col in binary_cols:
    if col in df.columns:
        df[col + "_encoded"] = df[col].map(binary_map).fillna(0)

internet_map = {"No": 0, "DSL": 1, "Fiber optic": 2}
df["InternetService_encoded"] = df["InternetService"].map(internet_map).fillna(0)

contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
df["Contract_encoded"] = df["Contract"].map(contract_map).fillna(0)

payment_map = {
    "Electronic check": 0, "Mailed check": 1,
    "Bank transfer (automatic)": 2, "Credit card (automatic)": 3
}
df["PaymentMethod_encoded"] = df["PaymentMethod"].map(payment_map).fillna(0)

service_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
service_map = {"No": 0, "Yes": 1, "No internet service": -1, "No phone service": -1}
for col in service_cols:
    df[col + "_encoded"] = df[col].map(service_map).fillna(0)

df["churn_risk_contract"] = (df["Contract_encoded"] == 0).astype(int)
df["churn_risk_internet"] = (df["InternetService_encoded"] == 2).astype(int)

print("Encoded features:")
print("  binary cols (Yes/No -> 1/0)")
print("  InternetService: No=0, DSL=1, Fiber optic=2")
print("  Contract: Month-to-month=0, One year=1, Two year=2")
print("  PaymentMethod: Electronic check=0, Mailed check=1, Bank transfer=2, Credit card=3")
print("  Service cols: No=0, Yes=1, No service=-1")
print()
print("Contract churn risk:")
print(f"  Month-to-month customers: {df['churn_risk_contract'].sum():,}")
print(f"  Churn rate among them   : {df[df['churn_risk_contract']==1]['Churn_Binary'].mean()*100:.1f}%")

ENCODING CATEGORICAL FEATURES

Encoded features:
  binary cols (Yes/No -> 1/0)
  InternetService: No=0, DSL=1, Fiber optic=2
  Contract: Month-to-month=0, One year=1, Two year=2
  PaymentMethod: Electronic check=0, Mailed check=1, Bank transfer=2, Credit card=3
  Service cols: No=0, Yes=1, No service=-1

Contract churn risk:
  Month-to-month customers: 3,875
  Churn rate among them   : 42.7%


In [3]:
# Financial and Behavioral Features
print("FINANCIAL AND BEHAVIORAL FEATURES")
print()

df["charges_per_month_tenure"] = df["MonthlyCharges"] / (df["tenure"] + 1)
df["total_charges_expected"] = df["MonthlyCharges"] * df["tenure"]
df["charges_deviation"] = df["TotalCharges"] - df["total_charges_expected"]
df["charges_deviation_pct"] = df["charges_deviation"] / (df["total_charges_expected"] + 1)
df["high_monthly_charges"] = (df["MonthlyCharges"] > df["MonthlyCharges"].quantile(0.75)).astype(int)
df["low_tenure"] = (df["tenure"] <= 12).astype(int)
df["long_tenure"] = (df["tenure"] >= 48).astype(int)
df["revenue_tier"] = pd.qcut(df["MonthlyCharges"], q=4, labels=[0,1,2,3]).astype(int)

services_encoded = [c for c in df.columns if c.endswith("_encoded") and
                    any(s in c for s in service_cols)]
df["total_services"] = df[[c for c in services_encoded if c != "InternetService_encoded"]].clip(0).sum(axis=1)
df["no_security_services"] = ((df["OnlineSecurity_encoded"] == 0) &
                               (df["DeviceProtection_encoded"] == 0) &
                               (df["OnlineBackup_encoded"] == 0)).astype(int)

df["customer_lifetime_value"] = df["TotalCharges"]
df["monthly_commitment_score"] = df["Contract_encoded"] * 2 + df["tenure"] / 12
df["churn_risk_score"] = (
    (1 - df["Contract_encoded"] / 2) * 0.35 +
    (df["InternetService_encoded"] == 2).astype(float) * 0.20 +
    (df["low_tenure"].astype(float)) * 0.25 +
    (df["high_monthly_charges"].astype(float)) * 0.10 +
    (1 - df["Partner_encoded"]) * 0.10
)

print("Financial features created:")
fin_feats = ["charges_per_month_tenure", "charges_deviation_pct",
             "high_monthly_charges", "low_tenure", "long_tenure",
             "total_services", "churn_risk_score", "monthly_commitment_score"]
for feat in fin_feats:
    corr = abs(df[feat].corr(df["Churn_Binary"]))
    churn_mean = df[df["Churn_Binary"]==1][feat].mean()
    retain_mean = df[df["Churn_Binary"]==0][feat].mean()
    print(f"  {feat:<30}: corr={corr:.4f} | churned={churn_mean:.3f} retained={retain_mean:.3f}")

FINANCIAL AND BEHAVIORAL FEATURES

Financial features created:
  charges_per_month_tenure      : corr=0.4118 | churned=11.746 retained=3.612
  charges_deviation_pct         : corr=0.0238 | churned=-0.001 retained=2.972
  high_monthly_charges          : corr=0.0828 | churned=0.309 retained=0.228
  low_tenure                    : corr=0.3176 | churned=0.555 retained=0.222
  long_tenure                   : corr=0.2668 | churned=0.119 retained=0.402
  total_services                : corr=0.0695 | churned=2.223 retained=2.545
  churn_risk_score              : corr=0.4791 | churned=0.698 retained=0.389
  monthly_commitment_score      : corr=0.4067 | churned=1.779 retained=4.909


In [4]:
#  Feature Importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

print("FEATURE IMPORTANCE RANKING")
print()

drop_cols = ["customerID", "gender", "Partner", "Dependents", "PhoneService",
             "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
             "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
             "Contract", "PaperlessBilling", "PaymentMethod", "Churn",
             "Churn_Binary"]

feature_cols = [c for c in df.columns if c not in drop_cols]
feature_cols = [c for c in feature_cols if df[c].dtype in ["float64","int64","float32","int32","int8"]]

X = df[feature_cols].fillna(0)
y = df["Churn_Binary"]

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1,
                             class_weight="balanced", max_depth=8)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
original_features = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]

print("Top 20 features:")
for i, (feat, imp) in enumerate(importances.head(20).items(), 1):
    tag = "ENGINEERED" if feat not in original_features else "original"
    print(f"  {i:2}. {feat:<35} {imp:.6f}  [{tag}]")

eng_imp = importances[[f for f in importances.index if f not in original_features]].sum()
orig_imp = importances[[f for f in importances.index if f in original_features]].sum()
total = eng_imp + orig_imp
print()
print(f"Engineered importance: {eng_imp/total*100:.1f}%")
print(f"Original importance  : {orig_imp/total*100:.1f}%")

FEATURE IMPORTANCE RANKING

Top 20 features:
   1. Churn_encoded                       0.585911  [ENGINEERED]
   2. churn_risk_score                    0.075763  [ENGINEERED]
   3. charges_per_month_tenure            0.037406  [ENGINEERED]
   4. monthly_commitment_score            0.036965  [ENGINEERED]
   5. churn_risk_contract                 0.036345  [ENGINEERED]
   6. Contract_encoded                    0.034102  [ENGINEERED]
   7. churn_risk_internet                 0.029721  [ENGINEERED]
   8. tenure                              0.022164  [original]
   9. total_charges_expected              0.012230  [ENGINEERED]
  10. MonthlyCharges                      0.011908  [original]
  11. customer_lifetime_value             0.011532  [ENGINEERED]
  12. TotalCharges                        0.010772  [original]
  13. no_security_services                0.009107  [ENGINEERED]
  14. InternetService_encoded             0.009011  [ENGINEERED]
  15. low_tenure                          0.008280 

In [5]:
# Save Engineered Data
print("SAVING ENGINEERED DATASET")
print()

drop_raw = ["customerID", "gender", "Partner", "Dependents", "PhoneService",
            "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
            "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
            "Contract", "PaperlessBilling", "PaymentMethod", "Churn"]

df_engineered = df.drop(columns=[c for c in drop_raw if c in df.columns])
df_engineered = df_engineered.drop(columns=["customerID"], errors="ignore")
df_engineered = df_engineered.fillna(0)

df_engineered.to_csv("../../data/customer_churn/churn_engineered.csv", index=False)

meta["feature_cols"] = feature_cols
meta["engineered_shape"] = list(df_engineered.shape)
with open("../../data/customer_churn/meta.json", "w") as f:
    json.dump(meta, f, indent=4)

print(f"Engineered shape : {df_engineered.shape}")
print(f"Features         : {len(feature_cols)}")
print(f"Churn rate       : {df_engineered['Churn_Binary'].mean()*100:.2f}%")
print()
print("Top 5 features by importance:")
for feat, imp in importances.head(5).items():
    print(f"  {feat:<35}: {imp:.6f}")
print()


SAVING ENGINEERED DATASET

Engineered shape : (7043, 36)
Features         : 35
Churn rate       : 26.54%

Top 5 features by importance:
  Churn_encoded                      : 0.585911
  churn_risk_score                   : 0.075763
  charges_per_month_tenure           : 0.037406
  monthly_commitment_score           : 0.036965
  churn_risk_contract                : 0.036345

